# Legoin – Tahap 2: Klasifikasi Jenis Part dengan MobileNetV3

**Kelompok 5 – Visual Komputer Cerdas** (Dika Aprilio Wibowo, Aldio Lisafron, Ricky Bonardo)

Tahap 1 menemukan setiap kepingan sebagai satu kelas `lego` dan memotongnya (crop). Notebook ini melatih **MobileNetV3-Large** untuk memberi nama jenis part pada tiap crop, sesuai **13 part katalog** di kontrak API (`legoin-docs/api/contract.md`, bagian *The catalogue*). Model punya satu output tambahan, **`other`**, untuk kepingan di luar katalog.

**Dataset:** Boiński, T. M., Zaraziński, S., & Śledź, B. (2021). *LEGO bricks for training classification network* (versi 1.1). MOST Wiedzy, Gdańsk University of Technology. https://doi.org/10.34808/rcza-jy08 (lisensi CC BY 4.0, wajib dicantumkan sumbernya).

- 447 jenis part, satu folder per nomor design LEGO; ±53 ribu **foto nyata** dan ±568 ribu **render**
- Semua gambar berisi 1 kepingan berlatar putih; render sudah diaugmentasi agar mirip foto
- Zip ±6 GB, tetapi notebook ini hanya mengunduh folder yang dipakai (±350 MB untuk 13 part katalog) lewat *HTTP range request*

**Runtime yang disarankan:** Google Colab dengan GPU (*Runtime → Change runtime type → T4 GPU*).

**Alur:** Setup → Unduh subset data → Eksplorasi → Split → Augmentasi → Training → Kalibrasi confidence → Evaluasi foto nyata → Uji crop tahap 1 → Simpan model

## 1. Setup lingkungan

In [ ]:
# Versi dikunci supaya hasil antar anggota kelompok sama.
# Di Colab, baris ini memasang ulang PyTorch bila versi bawaannya berbeda (beberapa menit).
%pip install -q torch==2.15.0 torchvision==0.29.2 pandas matplotlib tqdm

In [ ]:
import os
import sys
from pathlib import Path

# Google Colab: data di disk Colab (cepat), hasil training di Google Drive (permanen).
# Lokal (VS Code/Jupyter): semua disimpan di folder notebook ini.
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content')
    SIMPAN_DIR = Path('/content/drive/MyDrive/VISKOM_Kelompok5')
else:
    ROOT = Path.cwd()
    SIMPAN_DIR = ROOT
SIMPAN_DIR.mkdir(parents=True, exist_ok=True)

# Folder dataset: LEGOIN_DATA_DIR bila diset (mis. ~/dev/legoin/ml-datasets), selain itu ROOT / 'data'.
DATA_ROOT = Path(os.environ.get('LEGOIN_DATA_DIR') or ROOT / 'data').expanduser()
DATA_DIR = DATA_ROOT / 'lego_klasifikasi'       # subset dataset 447 kelas (hanya folder yang dipakai)
RUNS_DIR = SIMPAN_DIR / 'runs'                  # log & bobot hasil training (sama dengan tahap 1)
LATIH_DIR = RUNS_DIR / 'mobilenetv3_part'
LATIH_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import torch
import torchvision

print('torch', torch.__version__, '| torchvision', torchvision.__version__)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('GPU:', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda'
      else 'tidak ada (training di CPU akan sangat lambat)')

## 2. Konfigurasi

Semua angka yang biasa diubah saat eksperimen dikumpulkan di sini. Urutan `KATALOG` adalah urutan output model, dan nama kelasnya **persis id part di kontrak API**, supaya backend tidak perlu menerjemahkan apa pun.

In [ ]:
SEED = 42

# Katalog MVP dari kontrak API: id part -> design id LEGO
KATALOG = {
    'brick-1x1': '3005', 'brick-1x2': '3004', 'brick-1x3': '3622', 'brick-1x4': '3010',
    'brick-2x2': '3003', 'brick-2x3': '3002', 'brick-2x4': '3001',
    'plate-1x2': '3023', 'plate-2x2': '3022', 'plate-2x4': '3020',
    'slope-2x2': '3039', 'tile-1x2': '3069b', 'round-1x1': '3062b',
}
FOLDER_DATASET = {'3062b': '3062'}   # nama folder di dataset bila beda dari design id
KELAS_LAIN = 'other'                 # output ke-14: part di luar katalog, tidak pernah dikirim ke web
PAKAI_KELAS_LAIN = True

IMGSZ = 224                    # ukuran input MobileNetV3 (pretrained ImageNet)
EPOCHS = 15
BATCH = 64
LR = 5e-4
PATIENCE = 4                   # early stopping bila akurasi val tidak membaik
NUM_WORKERS = 2 if sys.platform == 'linux' else 0   # Colab punya 2 core; di macOS/Windows harus 0
PAKAI_RENDER = True            # render hanya masuk train; val dan test murni foto nyata
MAKS_FOTO_PER_KELAS = None     # None = semua foto
MAKS_RENDER_PER_KELAS = 1500   # dibatasi supaya kelas seimbang dan training cepat
RASIO_SPLIT = (0.70, 0.15, 0.15)   # train / val / test foto nyata, dibagi per foto asli

# Part di luar katalog untuk kelas `other`: yang mirip part katalog diutamakan,
# sisanya diambil acak dari 434 jenis lain sampai LAIN_N_FOLDER.
PART_MIRIP = ['3024', '3623', '3710', '3666', '3460', '3021', '3795', '3034', '3009', '3008',
              '3007', '2357', '3700', '4070', '6091', '3941', '3040b', '3038', '3037', '3298',
              '3045', '3665', '3660', '85984', '3070b', '63864', '2431', '87079', '98138', '6141',
              '2412b']
LAIN_N_FOLDER = 60
LAIN_FOTO_PER_FOLDER = 10
LAIN_RENDER_PER_FOLDER = 25

BATAS_YAKIN = 0.6   # LOW_CONFIDENCE di web app: kepingan di bawah ini ditandai untuk dicek

## 3. Unduh subset dataset

Zip dataset berukuran ±6 GB, padahal kita hanya butuh 13 dari 447 folder. Server MOST Wiedzy (S3) mendukung **HTTP range request**, jadi:
1. `zipfile` membaca daftar isi zip (central directory, beberapa MB di akhir file) lewat objek `BerkasHTTP` yang bisa di-*seek*;
2. isi tiap folder tersimpan berurutan di dalam zip, sehingga satu folder cukup diunduh dengan **satu request** rentang byte;
3. tiap gambar di-resize ke sisi terpanjang 256 px lalu disimpan sebagai JPEG.

Gambar yang sudah ada dilewati, jadi cell ini aman dijalankan ulang. Notebook tahap 3 memakai folder data yang sama.

In [ ]:
import io
import struct
import time
import urllib.request
import zipfile
import zlib
from bisect import bisect_right
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

from PIL import Image, ImageOps
from tqdm.auto import tqdm

URL_DATASET = ('https://mostwiedzy.pl/en/open-research-data/'
               'lego-bricks-for-training-classification-network,202309140842198941751-0/download')
UKURAN_SIMPAN = 256   # sisi terpanjang gambar yang disimpan (cukup untuk input 224)


def minta_rentang(url, awal, akhir, coba=5):
    # GET dengan header Range, diulang bila koneksi terputus.
    for i in range(coba):
        try:
            req = urllib.request.Request(url, headers={'Range': f'bytes={awal}-{akhir}'})
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.geturl(), r.headers.get('Content-Range'), r.read()
        except OSError:
            if i == coba - 1:
                raise
            time.sleep(2 ** i)


class BerkasHTTP:
    # Berkas jarak jauh yang bisa di-seek, cukup untuk zipfile membaca daftar isi zip.

    def __init__(self, url):
        self.url, rentang, _ = minta_rentang(url, 0, 0)   # url S3 setelah redirect (berlaku 24 jam)
        self.size = int(rentang.split('/')[-1])
        self.pos = 0

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        self.pos = (offset, self.pos + offset, self.size + offset)[whence]
        return self.pos

    def read(self, n=-1):
        n = self.size - self.pos if n is None or n < 0 else min(n, self.size - self.pos)
        if n <= 0:
            return b''
        data = minta_rentang(self.url, self.pos, self.pos + n - 1)[2]
        self.pos += len(data)
        return data


def isi_anggota(potongan, awal, info):
    # Ambil isi satu file dari potongan byte zip yang dimulai di offset `awal`.
    p = info.header_offset - awal
    n_nama, n_extra = struct.unpack('<2H', potongan[p + 26:p + 30])
    mulai = p + 30 + n_nama + n_extra
    data = potongan[mulai:mulai + info.compress_size]
    return zlib.decompress(data, -15) if info.compress_type == zipfile.ZIP_DEFLATED else data


def path_lokal(info):
    sumber, folder, nama = info.filename.split('/')
    return DATA_DIR / sumber / folder / (Path(nama).stem + '.jpg')


def simpan_gambar(data, tujuan):
    im = ImageOps.exif_transpose(Image.open(io.BytesIO(data))).convert('RGB')
    im.thumbnail((UKURAN_SIMPAN, UKURAN_SIMPAN), Image.LANCZOS)
    tujuan.parent.mkdir(parents=True, exist_ok=True)
    im.save(tujuan, quality=92)


print('Membaca daftar isi zip ...')
berkas_zip = BerkasHTTP(URL_DATASET)
daftar_zip = zipfile.ZipFile(berkas_zip)
ISI_FOLDER = defaultdict(list)   # (photos|renders, folder) -> [ZipInfo] dalam urutan zip
for info in daftar_zip.infolist():
    bagian = info.filename.split('/')
    if len(bagian) == 3 and bagian[2]:
        ISI_FOLDER[(bagian[0], bagian[1])].append(info)
OFFSET_URUT = sorted(i.header_offset for infos in ISI_FOLDER.values() for i in infos) + [daftar_zip.start_dir]
print(f'{berkas_zip.size / 1e9:.1f} GB, {len(OFFSET_URUT) - 1} gambar,',
      len({f for _, f in ISI_FOLDER}), 'folder part')


def unduh_folder(anggota):
    # Unduh file-file `anggota` (satu folder) dengan satu request rentang byte.
    kurang = [i for i in anggota if not path_lokal(i).exists()]
    if not kurang:
        return 0
    kurang.sort(key=lambda i: i.header_offset)
    awal = kurang[0].header_offset
    akhir = OFFSET_URUT[bisect_right(OFFSET_URUT, kurang[-1].header_offset)]
    potongan = minta_rentang(berkas_zip.url, awal, akhir - 1)[2]
    for info in kurang:
        simpan_gambar(isi_anggota(potongan, awal, info), path_lokal(info))
    return len(kurang)


def unduh_semua(pilihan, desc):
    with ThreadPoolExecutor(max_workers=4) as ex:
        n = sum(tqdm(ex.map(unduh_folder, pilihan), total=len(pilihan), desc=desc))
    print(desc, ':', n, 'gambar baru diunduh')

In [ ]:
import random

import pandas as pd

rng = random.Random(SEED)
KELAS = list(KATALOG) + ([KELAS_LAIN] if PAKAI_KELAS_LAIN else [])
N_KATALOG = len(KATALOG)
IDX_LAIN = KELAS.index(KELAS_LAIN) if PAKAI_KELAS_LAIN else None

folder_katalog = {FOLDER_DATASET.get(d, d): part for part, d in KATALOG.items()}
hilang = [f for f in folder_katalog if ('photos', f) not in ISI_FOLDER]
assert not hilang, f'folder tidak ada di dataset: {hilang}'

# File yang dipakai: N pertama dalam urutan zip (urutannya sudah acak), supaya rentang unduhan pendek.
pilihan = {}
for folder in folder_katalog:
    pilihan[('photos', folder)] = ISI_FOLDER[('photos', folder)][:MAKS_FOTO_PER_KELAS]
    if PAKAI_RENDER:
        pilihan[('renders', folder)] = ISI_FOLDER[('renders', folder)][:MAKS_RENDER_PER_KELAS]

folder_lain = []
if PAKAI_KELAS_LAIN:
    kandidat = sorted({f for s, f in ISI_FOLDER if s == 'photos' and ('renders', f) in ISI_FOLDER}
                      - set(folder_katalog))
    mirip = [f for f in PART_MIRIP if f in kandidat][:LAIN_N_FOLDER]
    sisa = [f for f in kandidat if f not in mirip]
    folder_lain = mirip + rng.sample(sisa, max(0, LAIN_N_FOLDER - len(mirip)))
    for folder in folder_lain:
        pilihan[('photos', folder)] = ISI_FOLDER[('photos', folder)][:LAIN_FOTO_PER_FOLDER]
        pilihan[('renders', folder)] = ISI_FOLDER[('renders', folder)][:LAIN_RENDER_PER_FOLDER]

unduh_semua(list(pilihan.values()), 'unduh')

## 4. Eksplorasi data

In [ ]:
import re

rows = []
for (sumber, folder), infos in pilihan.items():
    for info in infos:
        nama = Path(info.filename).stem
        # Foto: c<k>_<n>_..._<timestamp 13 digit>. Crop c0, c1, ... dari satu foto asli punya timestamp sama.
        cap = re.search(r'\d{13}', nama)
        rows.append({'path': path_lokal(info), 'sumber': sumber, 'folder': folder,
                     'kelas': folder_katalog.get(folder, KELAS_LAIN),
                     'grup': f'{folder}/{cap.group(0) if cap else nama}'})
df = pd.DataFrame(rows)
df['y'] = df.kelas.map(KELAS.index)

pd.crosstab(df.kelas, df.sumber, margins=True).reindex(KELAS + ['All'])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 13, figsize=(26, 8.5))
for j, part in enumerate(KATALOG):
    for i, sumber in enumerate(['photos', 'photos', 'renders', 'renders']):
        ax = axes[i, j]
        ax.axis('off')
        pool = df[(df.kelas == part) & (df.sumber == sumber)]
        if len(pool) > i % 2:
            ax.imshow(Image.open(pool.path.iloc[i % 2]))
        if i == 0:
            ax.set_title(part, fontsize=10)
fig.suptitle('Baris 1–2: foto nyata, baris 3–4: render', y=1.0)
plt.tight_layout()
plt.show()

Foto nyata dan render sama-sama berlatar putih dan dipotong rapat di sekitar kepingan, mirip crop dari tahap 1. Bedanya, crop tahap 1 berasal dari foto HP dengan latar meja atau lantai. Karena itu augmentasi di bagian 6 mengganti latar putih dengan latar acak.

Warna kepingan acak (render memakai 43 warna). Ini bagus: model dipaksa mengenali **bentuk**, karena warna adalah tugas tahap 3.

## 5. Split train / val / test

- **Foto nyata** part katalog dibagi 70/15/15 per kelas. Satu foto asli berisi beberapa kepingan sejenis yang di-crop menjadi `c0`, `c1`, …, jadi pembagian dilakukan **per foto asli** (grup), supaya crop dari foto yang sama tidak ada di train dan test sekaligus.
- **Render** hanya masuk **train**, seperti di tahap 1, supaya evaluasi murni pada foto nyata.
- **Part di luar katalog** dibagi **per jenis part**: part di val dan test tidak pernah dilihat model saat training. Dengan begitu test mengukur hal yang terjadi di aplikasi, yaitu kepingan asing yang belum pernah dilihat.

In [ ]:
def bagi(kunci, rasio=RASIO_SPLIT):
    # Acak lalu bagi daftar kunci menjadi {kunci: split}; val dan test minimal 1 bila kunci >= 3.
    kunci = sorted(kunci)
    rng.shuffle(kunci)
    n = len(kunci)
    n_va = max(1, round(n * rasio[1])) if n >= 3 else 0
    n_te = max(1, round(n * rasio[2])) if n >= 3 else 0
    n_tr = n - n_va - n_te
    return {k: 'train' if i < n_tr else 'val' if i < n_tr + n_va else 'test' for i, k in enumerate(kunci)}


split = {}
for part in KATALOG:
    foto = df[(df.kelas == part) & (df.sumber == 'photos')]
    peta = bagi(foto.grup.unique())
    split.update({i: peta[g] for i, g in zip(foto.index, foto.grup)})
split.update({i: 'train' for i in df[(df.kelas != KELAS_LAIN) & (df.sumber == 'renders')].index})

peta_lain = bagi(folder_lain)
for i, r in df[df.kelas == KELAS_LAIN].iterrows():
    s = peta_lain[r.folder]
    if s == 'train' or r.sumber == 'photos':   # val/test `other` hanya foto nyata
        split[i] = s
df['split'] = df.index.map(split)
df = df[df['split'].notna()].reset_index(drop=True)

pd.crosstab([df.sumber, df.kelas == KELAS_LAIN], df['split'], margins=True).rename_axis(['sumber', 'other'])

## 6. Dataset dan augmentasi

**Preprocessing yang juga wajib dipakai backend:** crop di-*letterbox* ke persegi (sisi terpanjang jadi 224 px, sisanya diisi warna `(124, 116, 104)`), lalu dinormalisasi dengan mean/std ImageNet. Gambar **tidak** di-stretch, karena 1×2 dan 1×4 terutama dibedakan dari rasio panjang-lebarnya.

**Augmentasi train:**
- ganti latar putih dengan latar acak (warna polos, gradasi, derau), untuk mendekati crop foto HP;
- crop dan rotasi ringan, flip, blur (foto nyata kadang buram);
- *color jitter* kuat sampai hue penuh dan sesekali grayscale, supaya model tidak menebak part dari warnanya.

In [ ]:
import numpy as np
import torchvision.transforms as T

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
WARNA_PAD = tuple(round(m * 255) for m in MEAN)   # (124, 116, 104): bernilai 0 setelah normalisasi


def letterbox(im, ukuran=IMGSZ, isi=WARNA_PAD):
    # Sisi terpanjang = ukuran, sisanya diisi warna `isi` sampai persegi (rasio sisi tetap).
    im = ImageOps.contain(im, (ukuran, ukuran), Image.BILINEAR)
    kanvas = Image.new('RGB', (ukuran, ukuran), isi)
    kanvas.paste(im, ((ukuran - im.width) // 2, (ukuran - im.height) // 2))
    return kanvas


def latar_acak(h, w, g):
    warna, warna2 = g.integers(0, 256, 3), g.integers(0, 256, 3)
    jenis = g.integers(3)
    if jenis == 0:
        return np.broadcast_to(warna, (h, w, 3)).astype(np.float32)
    if jenis == 1:
        t = np.linspace(0, 1, w)[None, :, None] if g.random() < 0.5 else np.linspace(0, 1, h)[:, None, None]
        return warna * (1 - t) + warna2 * t + np.zeros((h, w, 1))
    return warna + g.normal(0, 25, (h, w, 3))


class GantiLatar:
    # Ganti piksel yang mirip warna tepi gambar (latar putih) dengan latar acak.

    def __init__(self, p=0.5, ambang=40):
        self.p, self.ambang = p, ambang

    def __call__(self, im):
        if random.random() > self.p:
            return im
        a = np.asarray(im, dtype=np.float32)
        h, w = a.shape[:2]
        latar = np.median(np.concatenate([a[0], a[-1], a[:, 0], a[:, -1]]), axis=0)
        tengah = np.median(a[h // 4:3 * h // 4, w // 4:3 * w // 4].reshape(-1, 3), axis=0)
        if np.abs(tengah - latar).max() < self.ambang:
            return im   # kepingan mirip latar (mis. putih di atas putih): biarkan
        mask = np.abs(a - latar).max(axis=2) < self.ambang
        g = np.random.default_rng(random.getrandbits(32))
        a = np.where(mask[..., None], latar_acak(h, w, g), a)
        return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))


tf_latih = T.Compose([
    GantiLatar(p=0.5),
    T.Lambda(letterbox),
    T.RandomResizedCrop(IMGSZ, scale=(0.75, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(15, fill=WARNA_PAD),
    T.ColorJitter(0.4, 0.4, 0.4, 0.5),
    T.RandomGrayscale(0.1),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.2),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
tf_uji = T.Compose([T.Lambda(letterbox), T.ToTensor(), T.Normalize(MEAN, STD)])


class DataLego(torch.utils.data.Dataset):
    def __init__(self, d, tf):
        self.paths, self.y, self.tf = d.path.tolist(), d.y.tolist(), tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert('RGB')), self.y[i]


def loader(nama_split, tf, acak=False):
    return torch.utils.data.DataLoader(
        DataLego(df[df['split'] == nama_split], tf), batch_size=BATCH, shuffle=acak, num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == 'cuda', drop_last=acak, persistent_workers=NUM_WORKERS > 0)


random.seed(SEED)
torch.manual_seed(SEED)
dl_train, dl_val, dl_test = loader('train', tf_latih, acak=True), loader('val', tf_uji), loader('test', tf_uji)

contoh = df[df['split'] == 'train'].sample(8, random_state=SEED)
fig, axes = plt.subplots(2, 8, figsize=(20, 5.5))
for k, r in enumerate(contoh.itertuples()):
    asli = Image.open(r.path).convert('RGB')
    x = tf_latih(asli).permute(1, 2, 0).numpy() * STD + MEAN
    axes[0, k].imshow(letterbox(asli))
    axes[1, k].imshow(np.clip(x, 0, 1))
    axes[0, k].set_title(r.kelas, fontsize=9)
for ax in axes.flat:
    ax.axis('off')
fig.suptitle('Atas: setelah letterbox (input val/test), bawah: setelah augmentasi train')
plt.tight_layout()
plt.show()

## 7. Training MobileNetV3-Large

Bobot awal `IMAGENET1K_V2` (pretrained ImageNet) dari torchvision, lalu lapisan terakhir diganti menjadi 13 + 1 output dan seluruh jaringan di-fine-tune (*transfer learning*). Optimizer AdamW dengan jadwal *one-cycle*, *label smoothing* 0,1, dan *mixed precision* di GPU. Bobot dengan akurasi val terbaik disimpan.

In [ ]:
import torch.nn.functional as F
from torchvision.models import MobileNet_V3_Large_Weights, mobilenet_v3_large


def buat_model(n_output, pretrained=True):
    model = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2 if pretrained else None)
    model.classifier[3] = torch.nn.Linear(model.classifier[3].in_features, n_output)
    return model


def prediksi(model, dl):
    # Logit semua gambar di loader (tanpa augmentasi) beserta label sebenarnya.
    model.eval()
    hasil, label = [], []
    with torch.no_grad(), torch.autocast(DEVICE.type, enabled=DEVICE.type == 'cuda'):
        for x, y in dl:
            hasil.append(model(x.to(DEVICE)).float().cpu())
            label.append(y)
    return torch.cat(hasil), torch.cat(label)


model = buat_model(len(KELAS)).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(dl_train), pct_start=0.1)
scaler = torch.amp.GradScaler(DEVICE.type, enabled=DEVICE.type == 'cuda')
BEST = LATIH_DIR / 'best.pt'

riwayat, terbaik, sabar = [], -1.0, 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, n = 0.0, 0
    for x, y in tqdm(dl_train, desc=f'epoch {epoch}', leave=False):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == 'cuda'):
            loss = F.cross_entropy(model(x), y, label_smoothing=0.1)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        total_loss, n = total_loss + loss.item() * len(y), n + len(y)
    logit_val, y_val = prediksi(model, dl_val)
    akurasi_val = (logit_val.argmax(1) == y_val).float().mean().item()
    riwayat.append({'epoch': epoch, 'loss_train': total_loss / n,
                    'loss_val': F.cross_entropy(logit_val, y_val).item(), 'akurasi_val': akurasi_val})
    print(f"epoch {epoch}: loss train {riwayat[-1]['loss_train']:.3f} | akurasi val {akurasi_val:.1%}")
    if akurasi_val > terbaik:
        terbaik, sabar = akurasi_val, 0
        torch.save(model.state_dict(), BEST)
    else:
        sabar += 1
        if sabar >= PATIENCE:
            print('early stopping')
            break

riwayat = pd.DataFrame(riwayat)
riwayat.to_csv(LATIH_DIR / 'riwayat.csv', index=False)
model.load_state_dict(torch.load(BEST, map_location=DEVICE))
print(f'Model terbaik: {BEST} (akurasi val {terbaik:.1%})')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
riwayat.plot(x='epoch', y=['loss_train', 'loss_val'], marker='o', ax=ax1, title='Loss')
riwayat.plot(x='epoch', y='akurasi_val', marker='o', ax=ax2, title='Akurasi val (foto nyata)')
plt.tight_layout()
plt.show()

## 8. Kalibrasi confidence

Web app menandai kepingan dengan confidence di bawah **0,6** untuk dicek. Softmax jaringan saraf biasanya terlalu yakin, jadi angka 0,6 belum tentu berarti "benar 60% dari waktu". **Temperature scaling** mencari satu angka `T` pada data val sehingga `softmax(logit / T)` terkalibrasi. `T` disimpan di label map dan wajib dipakai backend.

**Aturan untuk backend** (juga usulan jawaban *open question 6* di kontrak):
- part = output katalog dengan peluang terbesar (argmax dari 13 output pertama);
- confidence = peluang part itu dari softmax **seluruh 14 output** setelah dibagi `T`.

Jika model condong ke `other`, peluang part katalog otomatis kecil (di bawah 0,5), sehingga kepingan **tidak dibuang** tetapi dikirim dengan tebakan part terdekat dan **ditandai untuk dicek**. Kepingan yang ditemukan detektor tetap muncul di foto, jadi membuangnya diam-diam membuat jumlah kotak dan inventori tidak cocok; pengguna tetap bisa menghapusnya lewat koreksi.

In [ ]:
def cari_suhu(logit, y):
    log_t = torch.zeros(1, requires_grad=True)
    opt_t = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)

    def nll():
        opt_t.zero_grad()
        loss = F.cross_entropy(logit / log_t.exp(), y)
        loss.backward()
        return loss

    opt_t.step(nll)
    return float(log_t.detach().exp())


def putuskan(prob):
    # prob: [n, 14] setelah temperature. Kembalikan (indeks part katalog, confidence).
    conf, idx = prob[:, :N_KATALOG].max(1)
    return idx, conf


logit_val, y_val = prediksi(model, dl_val)
SUHU = cari_suhu(logit_val, y_val)
print(f'T = {SUHU:.3f} | NLL val sebelum {F.cross_entropy(logit_val, y_val):.3f}, '
      f'sesudah {F.cross_entropy(logit_val / SUHU, y_val):.3f}')

## 9. Evaluasi pada data test (foto nyata)

**Penting dibaca sebelum melapor angka.** Target PRD adalah akurasi top-1 ≥ 85% pada **crop foto nyata**, yaitu crop dari foto HP pengguna dengan latar dan cahaya acak. Foto test di sini memang foto nyata, tetapi diambil di **studio berlatar putih** dengan cahaya terkontrol. Angka di bawah adalah **batas atas yang optimistis**, belum bukti target PRD tercapai. Bagian 10 menjelaskan cara mengukur target yang sebenarnya.

In [ ]:
import math


def wilson(k, n, z=1.96):
    # Selang kepercayaan 95% untuk proporsi k/n.
    if n == 0:
        return (float('nan'), float('nan'))
    p, d = k / n, 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)


logit_test, y_test = prediksi(model, dl_test)
prob_test = F.softmax(logit_test / SUHU, 1)
idx_test, conf_test = putuskan(prob_test)
pred_penuh = prob_test.argmax(1)
uji = df[df['split'] == 'test'].reset_index(drop=True).assign(
    pred=[KELAS[i] for i in pred_penuh], kirim=[KELAS[i] for i in idx_test], conf=conf_test.numpy())

kat = uji[uji.kelas != KELAS_LAIN]
k_benar = int((kat.pred == kat.kelas).sum())
lo, hi = wilson(k_benar, len(kat))
yakin = kat[kat.conf >= BATAS_YAKIN]
metrik_part = {
    'foto test (part katalog)': len(kat),
    'akurasi top-1 (14 output)': k_benar / len(kat),
    'CI 95% bawah': lo, 'CI 95% atas': hi,
    'akurasi part yang dikirim (13 output)': (kat.kirim == kat.kelas).mean(),
    f'porsi conf >= {BATAS_YAKIN} (tidak ditandai)': len(yakin) / len(kat),
    f'akurasi saat conf >= {BATAS_YAKIN}': (yakin.kirim == yakin.kelas).mean() if len(yakin) else float('nan'),
}
lain = uji[uji.kelas == KELAS_LAIN]
if len(lain):
    metrik_part['foto test part asing (belum pernah dilihat)'] = len(lain)
    metrik_part['part asing diprediksi `other`'] = (lain.pred == KELAS_LAIN).mean()
    metrik_part[f'part asing ditandai (conf < {BATAS_YAKIN})'] = (lain.conf < BATAS_YAKIN).mean()
pd.Series(metrik_part).round(3)

In [ ]:
salah = kat[kat.pred != kat.kelas]
per_kelas = pd.DataFrame({
    'foto': kat.groupby('kelas').size(),
    'recall': (kat.pred == kat.kelas).groupby(kat.kelas).mean(),
    'salah_terbanyak': salah.groupby('kelas').pred.agg(lambda s: s.mode().iloc[0]),
}).reindex(list(KATALOG)).fillna({'salah_terbanyak': '-'})
per_kelas.round(3)

In [ ]:
cm = pd.crosstab(uji.kelas, uji.pred).reindex(index=KELAS, columns=KELAS, fill_value=0)
cm_norm = cm.div(cm.sum(axis=1).replace(0, 1), axis=0)
fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(KELAS)), KELAS, rotation=90)
ax.set_yticks(range(len(KELAS)), KELAS)
for i in range(len(KELAS)):
    for j in range(len(KELAS)):
        if cm.iat[i, j]:
            ax.text(j, i, cm.iat[i, j], ha='center', va='center', fontsize=8,
                    color='white' if cm_norm.iat[i, j] > 0.5 else 'black')
ax.set_xlabel('prediksi')
ax.set_ylabel('sebenarnya')
ax.set_title('Confusion matrix data test (jumlah foto)')
plt.tight_layout()
plt.show()

### 9.1 Risiko PRD: brick vs plate dari atas

Dari atas, brick dan plate dengan ukuran stud yang sama terlihat hampir identik; bedanya hanya tinggi (brick tiga kali plate). Tabel berikut mengelompokkan part per keluarga untuk melihat seberapa sering keluarga tertukar.

In [ ]:
def keluarga(k):
    return k.split('-')[0]


pd.crosstab(kat.kelas.map(keluarga), kat.pred.map(keluarga), margins=True)

## 10. Crop foto nyata dari HP (target PRD)

Belum ada data publik berisi crop foto HP yang dilabeli dengan 13 part katalog. Dataset tahap 1 (*Tagged images with LEGO bricks*) hanya berlabel `lego`, tanpa jenis part. Karena itu target PRD **belum bisa diukur**, dan tim perlu melabeli sendiri satu set kecil.

**Rencana pelabelan (sekali kerja untuk tahap 2 dan 3):**
1. Foto kepingan milik kelompok dengan HP: berbagai latar (meja kayu, lantai, kain), cahaya siang dan lampu, sebagian sedikit miring (mitigasi brick vs plate di PRD). Target ±30 crop per part katalog (±400 crop) plus ±50 kepingan di luar katalog.
2. Jalankan detektor tahap 1 dengan `save_crop=True`, lalu salin crop ke `data/foto_mandiri/crops/`.
3. Isi `data/foto_mandiri/label.csv` dengan kolom `file,part,color`: `file` relatif terhadap folder CSV, `part` id part kontrak atau `other`, `color` id warna kontrak atau `other`. Crop yang meragukan (terpotong, bertumpuk) diberi `part` kosong dan dilewati.
4. Dua anggota melabeli secara terpisah, lalu selisihnya dibahas; foto dari satu sesi pemotretan masuk ke split yang sama.
5. Jalankan cell di bawah. Bila akurasinya jauh di bawah studio, pakai sebagian crop ini untuk fine-tuning dan sisanya tetap untuk test.

Cell berikut mengevaluasi `label.csv` bila ada. Bila belum ada, cell menampilkan prediksi (tanpa label) pada crop hasil notebook tahap 1 sebagai pemeriksaan visual.

In [ ]:
LABEL_CSV = DATA_ROOT / 'foto_mandiri' / 'label.csv'
CROP_TAHAP1 = RUNS_DIR / 'prediksi_contoh' / 'crops'   # hasil bagian 10 notebook tahap 1


def klasifikasi_file(paths):
    d = pd.DataFrame({'path': list(paths), 'y': 0})
    dl = torch.utils.data.DataLoader(DataLego(d, tf_uji), batch_size=BATCH, num_workers=NUM_WORKERS)
    idx, conf = putuskan(F.softmax(prediksi(model, dl)[0] / SUHU, 1))
    return [KELAS[i] for i in idx], conf.numpy()


if LABEL_CSV.exists():
    lab = pd.read_csv(LABEL_CSV).dropna(subset=['part'])
    lab['pred'], lab['conf'] = klasifikasi_file([LABEL_CSV.parent / f for f in lab.file])
    dalam = lab[lab.part != KELAS_LAIN]
    k = int((dalam.pred == dalam.part).sum())
    print(f'Crop foto HP berlabel: {len(dalam)} | akurasi top-1 {k / len(dalam):.1%} '
          f'(CI 95% {wilson(k, len(dalam))[0]:.1%}–{wilson(k, len(dalam))[1]:.1%})')
    luar = lab[lab.part == KELAS_LAIN]
    if len(luar):
        print(f'Kepingan luar katalog ditandai (conf < {BATAS_YAKIN}): {(luar.conf < BATAS_YAKIN).mean():.1%}')
elif CROP_TAHAP1.exists():
    crops = sorted(CROP_TAHAP1.rglob('*.jpg'))[:12]
    pred, conf = klasifikasi_file(crops)
    fig, axes = plt.subplots(2, 6, figsize=(18, 6.5))
    for ax in axes.flat:
        ax.axis('off')
    for ax, p, k, c in zip(axes.flat, crops, pred, conf):
        ax.imshow(Image.open(p))
        ax.set_title(f'{k} ({c:.2f})', color='black' if c >= BATAS_YAKIN else 'red')
    fig.suptitle('Crop tahap 1 tanpa label: merah = ditandai untuk dicek')
    plt.tight_layout()
    plt.show()
else:
    print('Belum ada label.csv maupun crop tahap 1; akurasi pada crop foto HP belum bisa diukur.')

## 11. Simpan model dan label map

Backend memuat dua berkas:
- `legoin_mobilenetv3_part.pt`: `state_dict` untuk `torchvision.models.mobilenet_v3_large` dengan lapisan terakhir 14 output;
- `legoin_mobilenetv3_part_labels.json`: urutan kelas (persis id part kontrak), design id, indeks `other`, ukuran input, cara resize, normalisasi, temperature, dan batas confidence.

In [ ]:
import json
import shutil

BOBOT = SIMPAN_DIR / 'legoin_mobilenetv3_part.pt'
LABEL_MAP = SIMPAN_DIR / 'legoin_mobilenetv3_part_labels.json'
shutil.copy(BEST, BOBOT)

label_map = {
    'architecture': 'torchvision.models.mobilenet_v3_large',
    'weights_file': BOBOT.name,
    'torchvision': torchvision.__version__,
    'classes': list(KATALOG),                      # output 0..12 = id part kontrak API
    'design_ids': dict(KATALOG),
    'num_outputs': len(KELAS),
    'other_index': IDX_LAIN,                       # output tambahan, tidak pernah dikirim ke web
    'input_size': IMGSZ,
    'resize': 'letterbox: longest side to input_size (bilinear), pad to a square with pad_color, centred',
    'pad_color': list(WARNA_PAD),
    'mean': list(MEAN),
    'std': list(STD),
    'temperature': round(SUHU, 4),
    'decision': 'part = argmax of outputs 0..12; confidence = softmax(logits / temperature)[part] '
                'over all outputs',
    'low_confidence': BATAS_YAKIN,
}
LABEL_MAP.write_text(json.dumps(label_map, indent=2))
print('Bobot   :', BOBOT)
print('Label map:', LABEL_MAP)

# Cek: muat ulang seperti backend, hasilnya harus sama
cek = buat_model(label_map['num_outputs'], pretrained=False)
cek.load_state_dict(torch.load(BOBOT, map_location='cpu'))
cek.eval()
x = tf_uji(Image.open(kat.path.iloc[0]).convert('RGB'))[None]
with torch.no_grad():
    p = F.softmax(cek(x) / label_map['temperature'], 1)
i, c = putuskan(p)
print('Contoh:', kat.kelas.iloc[0], '->', label_map['classes'][int(i)], f'(confidence {float(c):.2f})')

## 12. Catatan dan langkah berikutnya

- Catat ke laporan: akurasi top-1 dengan CI 95%, recall per part, confusion matrix, tabel brick vs plate, dan seberapa sering part asing ditandai (bagian 9). Sebutkan jelas bahwa test berupa **foto studio**, bukan crop foto HP.
- Target PRD (≥ 85% pada crop foto nyata) baru bisa diukur setelah `label.csv` dari foto mandiri ada (bagian 10).
- **B200C LEGO Classification** (Kaggle, CC BY-SA 4.0) belum dipakai karena unduhannya butuh akun Kaggle. Untuk mencobanya: simpan `KAGGLE_USERNAME` dan `KAGGLE_KEY` di *Colab secrets* (ikon kunci) atau environment variable, jangan pernah di notebook; unduh dengan `kagglehub.dataset_download(...)`; ambil folder yang namanya design id katalog; masukkan hanya ke **train**, seperti render MOST Wiedzy.
- Eksperimen yang layak dibandingkan: `PAKAI_KELAS_LAIN` True vs False, `GantiLatar` aktif vs tidak, `PAKAI_RENDER` True vs False, dan `mobilenet_v3_small` untuk aplikasi Android.
- Bila brick vs plate tetap sering tertukar, opsi dari PRD: foto sedikit miring, atau gabungkan kelasnya (perubahan kontrak).